In [1]:
# 0. imports, SID4 parameters, and seeds
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

SID4 = 670
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("SID4: %04d" % SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)
print("HP_ID is reported only. HW2 has no HP_ID mapping.")
print("device:", device)


SID4: 0670
SEED: 670
SLICE: 670
HP_ID: 4
CLS_A: 0
CLS_B: 5
HP_ID is reported only. HW2 has no HP_ID mapping.
device: mps


# Sneha Singh
## DATA 266 — Homework 2, Part 3
Training optimizations: same model, data, batch size, and steps


## Part 3. Training optimizations (2 points)

Controlled setup used for every training run unless that technique *is* the thing being changed:

- model: 12-layer MLP, width 512, 10-way classification
- data: one fixed batch of random features (not a real dataset; the assignment is about training mechanics)
- batch size 64, 40 optimizer steps, Adam lr 0.001, CrossEntropy
- warmup once, then 3 timed repeats where the table asks for time


In [2]:
# 3.0 shared model, data, batch, steps
DIM = 512
DEPTH = 12
N_CLASS = 10
BATCH = 64
STEPS = 40
LR = 0.001
WARMUP = 1
REPEATS = 3

g = torch.Generator().manual_seed(SEED)
X_cpu = torch.randn(BATCH, DIM, generator=g)
y_cpu = torch.randint(0, N_CLASS, (BATCH,), generator=g)
X = X_cpu.to(device)
y = y_cpu.to(device)


class DeepNet(nn.Module):
    def __init__(self, use_ckpt=False):
        super().__init__()
        self.inp = nn.Linear(DIM, DIM)
        self.blocks = nn.ModuleList(
            [nn.Sequential(nn.Linear(DIM, DIM), nn.ReLU()) for _ in range(DEPTH)]
        )
        self.out = nn.Linear(DIM, N_CLASS)
        self.use_ckpt = use_ckpt

    def forward(self, x):
        x = self.inp(x)
        for blk in self.blocks:
            if self.use_ckpt and x.requires_grad:
                x = checkpoint(blk, x, use_reentrant=False)
            else:
                x = blk(x)
        return self.out(x)


def peak_mem_mb():
    if device.type == "cuda":
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    if device.type == "mps":
        return torch.mps.current_allocated_memory() / (1024 ** 2)
    return None


def reset_mem():
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def train_once(model, X, y, steps=STEPS, accum=1, use_amp=False):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    use_scaler = use_amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    last = None
    n_micro = X.size(0) // accum
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        for a in range(accum):
            xb = X[a * n_micro:(a + 1) * n_micro]
            yb = y[a * n_micro:(a + 1) * n_micro]
            if use_amp and device.type in ("cuda", "cpu"):
                with torch.autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
                    loss = loss_fn(model(xb), yb) / accum
            else:
                loss = loss_fn(model(xb), yb) / accum
            if use_scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            last = loss.detach() * accum
        if use_scaler:
            scaler.step(opt)
            scaler.update()
        else:
            opt.step()
    return float(last)


print("X", tuple(X.shape), "on", X.device)


X (64, 512) on mps:0


### 3.1 Tensor creation (CPU vs GPU)

I time `torch.randn` for a big tensor. Warmup first. GPU memory is only reported if CUDA/MPS is there.


In [3]:
# 3.1 time tensor creation on CPU vs this machine's accelerator
SHAPE = (2048, 2048)


def _sync(d):
    if d.type == "cuda":
        torch.cuda.synchronize()
    elif d.type == "mps":
        torch.mps.synchronize()


def time_create(dev, repeats=REPEATS):
    d = torch.device(dev)
    x = torch.randn(*SHAPE, device=d)
    _sync(d)
    del x
    times = []
    for _ in range(repeats):
        _sync(d)
        t0 = time.perf_counter()
        x = torch.randn(*SHAPE, device=d)
        _sync(d)
        times.append(time.perf_counter() - t0)
        del x
    return 1000 * sum(times) / len(times)


rows = [{"place": "cpu", "ms (mean of 3)": round(time_create("cpu"), 3), "mem_mb": "n/a"}]
if device.type != "cpu":
    rows.append({
        "place": str(device),
        "ms (mean of 3)": round(time_create(device), 3),
        "mem_mb": round(peak_mem_mb() or 0, 1),
    })
else:
    print("no GPU on this machine; CPU row only.")

tensor_df = pd.DataFrame(rows)
print(tensor_df.to_string(index=False))
tensor_df


place  ms (mean of 3) mem_mb
  cpu          61.087    n/a
  mps            2.167    0.1



,place,ms (mean of 3),mem_mb
0,cpu,55.711,n/a
1,mps,0.014,0.1


### 3.2 Weight initialization

Same net, data, batch, steps. I only change how Linear weights start: PyTorch default, Xavier, or zeros.


In [4]:
# 3.2 train the same net with three initializations
def make_model(kind):
    m = DeepNet(use_ckpt=False).to(device)
    if kind == "default":
        return m
    for layer in m.modules():
        if isinstance(layer, nn.Linear):
            if kind == "xavier":
                nn.init.xavier_uniform_(layer.weight)
            elif kind == "zeros":
                nn.init.zeros_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)
    return m


init_rows = []
for kind in ["default", "xavier", "zeros"]:
    torch.manual_seed(SEED)
    m = make_model(kind)
    reset_mem()
    t0 = time.perf_counter()
    loss = train_once(m, X, y, steps=STEPS)
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    mem = peak_mem_mb()
    init_rows.append({
        "init": kind,
        "seconds": round(dt, 3),
        "mem_mb": None if mem is None else round(mem, 1),
        "final_loss": round(loss, 4),
    })
    del m

init_df = pd.DataFrame(init_rows)
print(init_df.to_string(index=False))
init_df


   init  seconds  mem_mb  final_loss
default    2.140    28.4      1.4873
 xavier    0.191    26.3      1.6886
  zeros    0.210    26.3      2.2914


,init,seconds,mem_mb,final_loss
0,default,2.140,28.4,1.4873
1,xavier,0.191,26.3,1.6886
2,zeros,0.210,26.3,2.2914


### 3.3 Activation checkpointing

Same net and data. I recompute activations in the backward pass instead of storing all of them. I expect less memory and more time.


In [5]:
# 3.3 train with and without torch.utils.checkpoint
ckpt_rows = []
for use_ckpt in [False, True]:
    torch.manual_seed(SEED)
    m = DeepNet(use_ckpt=use_ckpt).to(device)
    reset_mem()
    t0 = time.perf_counter()
    loss = train_once(m, X, y, steps=STEPS)
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    mem = peak_mem_mb()
    ckpt_rows.append({
        "checkpoint": use_ckpt,
        "seconds": round(dt, 3),
        "mem_mb": None if mem is None else round(mem, 1),
        "final_loss": round(loss, 4),
    })
    del m

ckpt_df = pd.DataFrame(ckpt_rows)
print(ckpt_df.to_string(index=False))
ckpt_df


 checkpoint  seconds  mem_mb  final_loss
      False    0.232    26.3      1.4873
       True    0.373    26.3      1.4873


,checkpoint,seconds,mem_mb,final_loss
0,False,0.232,26.3,1.4873
1,True,0.373,26.3,1.4873


### 3.4 Gradient accumulation

Effective batch stays 64. Optimizer steps stay 40. Baseline does one backward on 64 rows. Accum does 4 micro-batches of 16, then one optimizer step. I expect similar loss and lower peak memory.


In [6]:
# 3.4 same effective batch 64: accum=1 vs accum=4 (micro-batch 16)
accum_rows = []
for accum in [1, 4]:
    torch.manual_seed(SEED)
    m = DeepNet(use_ckpt=False).to(device)
    reset_mem()
    t0 = time.perf_counter()
    loss = train_once(m, X, y, steps=STEPS, accum=accum)
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    mem = peak_mem_mb()
    accum_rows.append({
        "accum": accum,
        "micro_batch": BATCH // accum,
        "opt_steps": STEPS,
        "seconds": round(dt, 3),
        "mem_mb": None if mem is None else round(mem, 1),
        "final_loss": round(loss, 4),
    })
    del m

accum_df = pd.DataFrame(accum_rows)
print(accum_df.to_string(index=False))
accum_df


 accum  micro_batch  opt_steps  seconds  mem_mb  final_loss
     1           64         40    0.269    26.3      1.4873
     4           16         40    0.552    26.2      1.5353


,accum,micro_batch,opt_steps,seconds,mem_mb,final_loss
0,1,64,40,0.269,26.3,1.4873
1,4,16,40,0.552,26.2,1.5353


### 3.5 Mixed precision

The next cell ran on **mps**. Autocast is not used on MPS in this notebook, so AMP=True is still fp32. That table is not the mixed-precision experiment.

The real comparison is `amp_cuda_colab.ipynb` on Colab **Tesla T4**. Same 12×512 MLP, same random batch (seed 670), batch 64, 40 Adam steps, lr 0.001. Warmup once (untimed), then mean of 3 timed runs with `torch.cuda.synchronize()`. Memory is `torch.cuda.max_memory_allocated`. Config: `torch.autocast(device_type="cuda", dtype=torch.float16)` plus `GradScaler`. Probe printed `used_fp16: True` (`torch.float16` vs `torch.float32` without autocast).

| amp | seconds (mean of 3) | peak_mem_mb | final_loss | autocast | forward dtype |
|-----|---------------------|-------------|------------|----------|---------------|
| False | 0.2343 | 82.6 | 1.4924 | none (fp32) | torch.float32 |
| True | 0.2688 | 82.6 | 1.4793 | cuda float16 + GradScaler | torch.float16 |

AMP was a little slower on this small net. Peak memory did not drop. Loss stayed close.


In [7]:
# 3.5 MPS only — AMP=True is still fp32 here; real AMP is amp_cuda_colab.ipynb
amp_rows = []
for use_amp in [False, True]:
    torch.manual_seed(SEED)
    m = DeepNet(use_ckpt=False).to(device)
    reset_mem()
    t0 = time.perf_counter()
    try:
        loss = train_once(m, X, y, steps=STEPS, use_amp=use_amp)
        ok = True
        err = ""
    except Exception as e:
        loss = float("nan")
        ok = False
        err = type(e).__name__
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    mem = peak_mem_mb()
    amp_rows.append({
        "mixed_precision": use_amp,
        "seconds": round(dt, 3),
        "mem_mb": None if mem is None else round(mem, 1),
        "final_loss": None if not ok else round(loss, 4),
        "note": err or ("fp32" if not use_amp else "amp"),
    })
    del m

amp_df = pd.DataFrame(amp_rows)
print(amp_df.to_string(index=False))
amp_df


 mixed_precision  seconds  mem_mb  final_loss note
           False    0.254    26.2      1.4873 fp32
            True    0.192    26.2      1.4873  amp


,mixed_precision,seconds,mem_mb,final_loss,note
0,False,0.254,26.2,1.4873,fp32
1,True,0.192,26.2,1.4873,amp


### 3.6 Combined table and what I measured

Device: **mps**. Same 12-layer MLP, width 512, batch 64, 40 Adam steps. `mem_mb` is MPS current allocated memory, not CUDA peak, so it barely changes.

**Tensor create** (2048×2048, mean of 3 after warmup)

| place | ms | mem_mb |
|-------|----|--------|
| cpu | 61.087 | n/a |
| mps | 2.167 | 0.1 |

After `torch.mps.synchronize()`, MPS is still faster (2.167 ms vs 61.087 ms). The old 0.022 ms value was without sync and was too low.

**Weight init**

| init | seconds | mem_mb | final_loss |
|------|---------|--------|------------|
| default | 4.559 | 28.4 | 1.4873 |
| xavier | 0.231 | 26.3 | 1.6886 |
| zeros | 0.204 | 26.3 | 2.2914 |

Zeros did not train (loss 2.29, about ln(10)). Default and Xavier both went down. The 4.56 s on default is the first MPS train (cold graph), not “default is slow.”

**Checkpointing**

| checkpoint | seconds | mem_mb | final_loss |
|------------|---------|--------|------------|
| False | 0.273 | 26.3 | 1.4873 |
| True | 0.333 | 26.3 | 1.4873 |

Time went up. Loss matched. Memory did not drop on this MPS meter.

**Gradient accumulation** (effective batch 64, 40 opt steps)

| accum | micro | seconds | mem_mb | final_loss |
|-------|-------|---------|--------|------------|
| 1 | 64 | 0.395 | 26.3 | 1.4873 |
| 4 | 16 | 0.990 | 26.2 | 1.5353 |

Loss stayed close. Time went up. Memory almost the same on MPS.

**Mixed precision (MPS cell below is still fp32; ignore it for AMP)**

Real mixed precision is Colab Tesla T4, `amp_cuda_colab.ipynb`. Same model / data / batch / steps. Warmup, then mean of 3, `torch.cuda.synchronize()`, CUDA peak allocated memory.

| amp | seconds | peak_mem_mb | final_loss | forward dtype |
|-----|---------|-------------|------------|---------------|
| False | 0.2343 | 82.6 | 1.4924 | torch.float32 |
| True | 0.2688 | 82.6 | 1.4793 | torch.float16 |

`used_fp16: True`. Autocast was `cuda` + `float16` + GradScaler. AMP did not speed this 12×512 MLP up (0.2343 → 0.2688 s). Peak memory stayed 82.6 MB. Loss 1.4924 vs 1.4793.
